## 39_Göteborgsalliansen 
* [#39](https://github.com/salgo60/ifkdb/issues/39)
* denna Notebook [39_Göteborgsalliansen](https://github.com/salgo60/ifkdb/blob/main/Notebook/39_Göteborgsalliansen.ipynb)
* rapport
  * [Göteborgsalliansen_20260310.html](http://salgo60.github.io/ifkdb/Notebook/Go%CC%88teborgsalliansen_20260310.html)
* MixandMatch katalog skapad [mix-n-match.toolforge.org/#/catalog/7706](https://mix-n-match.toolforge.org/#/catalog/7706)
  
* [Mix and match](https://meta.wikimedia.org/wiki/Mix%27n%27match)
   * [import](https://meta.wikimedia.org/wiki/Mix%27n%27match/Import)
      * [upload](https://mix-n-match.toolforge.org/#/import)

In [80]:
from datetime import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

BASE = "https://goteborgsalliansen.se"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 salgo60@man.com"
})

start_time = datetime.now()
print("Start:", start_time)

Start: 2026-03-11 21:10:17.025146


In [116]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

BASE = "https://goteborgsalliansen.se"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 salgo60@msn.com"})

import re
import requests
from urllib.parse import urljoin

BASE = "https://goteborgsalliansen.se"

def get_player_links():

    r = requests.get(BASE)
    html = r.text

    links = re.findall(r'/spelare/[A-Za-z0-9+%_\-]+', html)

    urls = [urljoin(BASE, l) for l in links]

    return sorted(set(urls))

def parse_description(desc):

    birth = None
    position = None
    club = None

    b = re.search(r"Född ([^.]+)", desc)
    if b:
        birth = b.group(1)

    p = re.search(r"Position:\s*([^\.]+)", desc)
    if p:
        position = p.group(1)

    c = re.search(r"spelare i ([^\.]+)", desc)
    if c:
        club = c.group(1)

    return birth, position, club


def scrape_player(url):

    r = session.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    title = soup.title.text.strip() if soup.title else None

    meta = soup.find("meta", {"name": "description"})
    description = meta["content"] if meta else None

    birth = None
    position = None
    club = None

    if description:
        birth, position, club = parse_description(description)

    name = None
    birth_year = None
    death_year = None

    if title:
        m = re.search(r"^(.*?) \((\d{4})-(\d{4})\)", title)
        if m:
            name = m.group(1)
            birth_year = m.group(2)
            death_year = m.group(3)
        else:
            name = title.split("(")[0].strip()

    return {
        "name": name,
        "birth_year": birth_year,
        "death_year": death_year,
        "birth_date": birth,
        "position": position,
        "club": club,
        "url": url
    }


def scrape_all_players():

    links = get_player_links()

    rows = []

    for url in tqdm(links):
        try:
            rows.append(scrape_player(url))
        except:
            pass

    return pd.DataFrame(rows)

In [117]:
player_urls = get_player_links()

print("Players found:", len(player_urls))

with ThreadPoolExecutor(max_workers=5) as ex:
    results = list(tqdm(ex.map(scrape_player, player_urls), total=len(player_urls)))

df = pd.DataFrame(results)

df.head()

print("Players found:", len(df))


Players found: 378


100%|█████████████████████████████████████████| 378/378 [01:19<00:00,  4.73it/s]

Players found: 378


,name,birth_year,death_year,birth_date,position,club,url
0,Åke Andersson,1917,1983,22 april 1917,Anfallare,GAIS för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...
1,Åke Bergling,None,None,4 februari 1943,Målvakt,Örgryte IS för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...
2,Åke Gustavsson,None,None,10 mars 1935,Mittfältare,IFK Göteborg för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...
3,Åke Hallman,1912,1973,12 november 1912,Anfallare,IFK Göteborg för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...
4,Åke Hansson,1903,1981,16 april 1903,Mittfältare,IFK Göteborg och GAIS för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...


In [129]:
from urllib.parse import unquote

df["external_id"] = (
    df["url"]
    .str.extract(r'/spelare/([^/]+)')[0]
    .apply(unquote)
)

In [130]:
df["name_clean"] = (
    df["external_id"]
    .str.replace(r"_\d+$","",regex=True)
    .str.replace("+"," ",regex=False)
)

In [133]:
df["label"] = (
    df["external_id"]
    .str.replace(r"_\d+$","",regex=True)
    .str.replace("+"," ",regex=False)
)

In [137]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         378 non-null    object
 1   birth_year   288 non-null    object
 2   death_year   288 non-null    object
 3   birth_date   376 non-null    object
 4   position     378 non-null    object
 5   club         378 non-null    object
 6   url          378 non-null    object
 7   external_id  378 non-null    object
 8   name_clean   378 non-null    object
 9   label        378 non-null    object
dtypes: object(10)
memory usage: 29.7+ KB


In [135]:
df.head()

,name,birth_year,death_year,birth_date,position,club,url,external_id,name_clean,label
0,Åke Andersson,1917,1983,22 april 1917,Anfallare,GAIS för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...,Åke+Andersson_139,Åke Andersson,Åke Andersson
1,Åke Bergling,None,None,4 februari 1943,Målvakt,Örgryte IS för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...,Åke+Bergling_296,Åke Bergling,Åke Bergling
2,Åke Gustavsson,None,None,10 mars 1935,Mittfältare,IFK Göteborg för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...,Åke+Gustavsson_276,Åke Gustavsson,Åke Gustavsson
3,Åke Hallman,1912,1973,12 november 1912,Anfallare,IFK Göteborg för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...,Åke+Hallman_162,Åke Hallman,Åke Hallman
4,Åke Hansson,1903,1981,16 april 1903,Mittfältare,IFK Göteborg och GAIS för Göteborgsalliansen,https://goteborgsalliansen.se/spelare/%C3%85ke...,Åke+Hansson_15,Åke Hansson,Åke Hansson


### Skippar SPARQL

In [109]:
import requests
import pandas as pd

url = "https://query.wikidata.org/sparql"

query = """
SELECT ?player ?playerLabel ?birth WHERE {
  VALUES ?team {
    wd:Q391789 # GAIS
    wd:Q201567 # IFK Göteborg
    wd:Q297906 # Örgryte IS 
    wd:Q1270457 # IFK Göteborg
    wd:Q11903335 # Örgryte IS 
    wd:Q1508540 # Gårda BK
    wd:Q10512920 # Göteborgsalliansen
    wd:Q337839 # Göteborgs FF
    wd:Q1654931 # IS Halmia 
    
  }

  ?player wdt:P54 ?team.
  optional {?player wdt:P569 ?birth}

  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv,en". }
}


"""


## Skapa HTMLrapport

## Skapa Mix and Match 



In [160]:
mnm = pd.DataFrame()

mnm["id"] = df["external_id"]
mnm["name"] = df["name"]

mnm["desc"] = (
    df["club"].fillna("") +
    " " +
    df["position"].fillna("") + 
    " född " + df["birth_date"].fillna("")    
).str.strip()

mnm["type"] = "Q5"
mnm["url"] = df["url"]

mnm.head()

,id,name,desc,type,url
0,Åke+Andersson_139,Åke Andersson,GAIS för Göteborgsalliansen Anfallare född 22 ...,Q5,https://goteborgsalliansen.se/spelare/%C3%85ke...
1,Åke+Bergling_296,Åke Bergling,Örgryte IS för Göteborgsalliansen Målvakt född...,Q5,https://goteborgsalliansen.se/spelare/%C3%85ke...
2,Åke+Gustavsson_276,Åke Gustavsson,IFK Göteborg för Göteborgsalliansen Mittfältar...,Q5,https://goteborgsalliansen.se/spelare/%C3%85ke...
3,Åke+Hallman_162,Åke Hallman,IFK Göteborg för Göteborgsalliansen Anfallare ...,Q5,https://goteborgsalliansen.se/spelare/%C3%85ke...
4,Åke+Hansson_15,Åke Hansson,IFK Göteborg och GAIS för Göteborgsalliansen M...,Q5,https://goteborgsalliansen.se/spelare/%C3%85ke...


In [161]:
mnm.isna().sum()

id      0
name    0
desc    0
type    0
url     0
dtype: int64

In [162]:
mnm.duplicated("id").sum()

0

In [163]:
from datetime import datetime

today = datetime.now().strftime("%Y%m%d")

filename = f"goteborgsalliansen_mixnmatch_{today}.csv"

mnm.to_csv(
    filename,
    index=False,
    encoding="utf-8"
)

print("Saved:", filename)



Saved: goteborgsalliansen_mixnmatch_20260311.csv


In [154]:
# End timer
end_time = datetime.now()
duration = end_time - start_time

print("\n--- Execution Summary ---")
print(f"End time:      {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Duration:      {duration}")
print(f"Total seconds: {duration.total_seconds():.2f}")
print(f"Python ver:    {sys.version.split()[0]}")
print(f"Platform:      {platform.system()} {platform.release()}")
print(f"Process ID:    {os.getpid()}")


--- Execution Summary ---
End time:      2026-03-11 23:04:18
Duration:      1:54:01.891340
Total seconds: 6841.89
Python ver:    3.12.2
Platform:      Darwin 25.3.0
Process ID:    85208
